In [31]:
from dotenv import load_dotenv
import os

# Load API key from .env file — key never appears in code
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY missing — check your .env file")

print(" API key loaded securely")

 API key loaded securely


In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('superstore.csv', encoding='latin-1')

# First look
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)


Shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First 3 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [8]:
# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Group by week and sum sales
df_weekly = df.groupby(pd.Grouper(key='Order Date', freq='W'))['Sales'].sum().reset_index()
df_weekly.columns = ['ds', 'y']  # Prophet requires exactly these column names

# Remove any weeks with zero sales
df_weekly = df_weekly[df_weekly['y'] > 0].reset_index(drop=True)

print("Weekly data shape:", df_weekly.shape)
print("\nDate range:", df_weekly['ds'].min(), "to", df_weekly['ds'].max())
print("\nSample:")
df_weekly.head(5)

Weekly data shape: (209, 2)

Date range: 2014-01-05 00:00:00 to 2017-12-31 00:00:00

Sample:


,ds,y
0,2014-01-05,324.044
1,2014-01-12,4599.572
2,2014-01-19,4509.127
3,2014-01-26,3842.388
4,2014-02-02,1642.310


In [5]:
%pip install prophet
from prophet import Prophet



Note: you may need to restart the kernel to use updated packages.


In [7]:
# Train Prophet model on our weekly sales
model = Prophet(
    changepoint_prior_scale=0.05,  # sensitivity to trend changes
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

model.fit(df_weekly)

# Forecast for the same period (in-sample) to detect anomalies
forecast = model.predict(df_weekly)

print("Forecast done.")
print("Columns generated:", forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(3))

22:50:30 - cmdstanpy - INFO - Chain [1] start processing
22:50:30 - cmdstanpy - INFO - Chain [1] done processing


Forecast done.
Columns generated:             ds          yhat    yhat_lower    yhat_upper
206 2017-12-17  22042.289875  15318.990073  28905.690343
207 2017-12-24  19418.477395  13102.476584  26104.291478
208 2017-12-31  15089.455744   8094.005406  21863.652491


In [9]:
# Merge actual sales with forecast
results = df_weekly[['ds', 'y']].copy()
results['yhat'] = forecast['yhat'].values
results['yhat_lower'] = forecast['yhat_lower'].values
results['yhat_upper'] = forecast['yhat_upper'].values

# Calculate deviation percentage from expected
results['deviation_pct'] = ((results['y'] - results['yhat']) / results['yhat'] * 100).round(2)

# Flag anomalies — actual sales fell BELOW the lower bound (unexpected drop)
# OR rose ABOVE the upper bound (unexpected spike)
results['is_anomaly'] = (
    (results['y'] < results['yhat_lower']) |   # drop anomaly
    (results['y'] > results['yhat_upper'])      # spike anomaly
)

results['anomaly_type'] = 'Normal'
results.loc[results['y'] < results['yhat_lower'], 'anomaly_type'] = 'DROP'
results.loc[results['y'] > results['yhat_upper'], 'anomaly_type'] = 'SPIKE'

# Show only anomalies
anomalies = results[results['is_anomaly'] == True].copy()
print(f"Total anomalies detected: {len(anomalies)}")
print(f"  - Drops : {len(anomalies[anomalies['anomaly_type'] == 'DROP'])}")
print(f"  - Spikes: {len(anomalies[anomalies['anomaly_type'] == 'SPIKE'])}")
print("\nTop 5 anomalies by deviation:")
anomalies[['ds', 'y', 'yhat', 'deviation_pct', 'anomaly_type']].sort_values('deviation_pct').head(5)

Total anomalies detected: 33
  - Drops : 13
  - Spikes: 20

Top 5 anomalies by deviation:


,ds,y,yhat,deviation_pct,anomaly_type
62,2015-03-15,2729.488,11277.171976,-75.80,DROP
116,2016-03-27,3739.864,14438.384956,-74.10,DROP
91,2015-10-04,4266.842,13443.782498,-68.26,DROP
199,2017-10-29,6423.353,15855.958330,-59.49,DROP
147,2016-10-30,6317.425,14821.679475,-57.38,DROP


In [10]:
%pip install plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'browser'


Note: you may need to restart the kernel to use updated packages.


In [11]:

fig = go.Figure()

# Confidence band (expected range)
fig.add_trace(go.Scatter(
    x=results['ds'].tolist() + results['ds'].tolist()[::-1],
    y=results['yhat_upper'].tolist() + results['yhat_lower'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(46, 117, 182, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Expected Range'
))

# Forecast line
fig.add_trace(go.Scatter(
    x=results['ds'], y=results['yhat'],
    mode='lines',
    line=dict(color='rgba(46, 117, 182, 0.6)', dash='dash', width=1.5),
    name='Forecast'
))

# Actual sales line
fig.add_trace(go.Scatter(
    x=results['ds'], y=results['y'],
    mode='lines',
    line=dict(color='#1F4E79', width=2),
    name='Actual Sales'
))

# DROP anomalies — red dots
drops = results[results['anomaly_type'] == 'DROP']
fig.add_trace(go.Scatter(
    x=drops['ds'], y=drops['y'],
    mode='markers',
    marker=dict(color='red', size=10, symbol='circle'),
    name='DROP Anomaly',
    text=drops['deviation_pct'].apply(lambda x: f"{x:.1f}%"),
    hovertemplate='<b>DROP</b><br>Date: %{x}<br>Sales: %{y:.0f}<br>Deviation: %{text}<extra></extra>'
))

# SPIKE anomalies — orange dots
spikes = results[results['anomaly_type'] == 'SPIKE']
fig.add_trace(go.Scatter(
    x=spikes['ds'], y=spikes['y'],
    mode='markers',
    marker=dict(color='orange', size=10, symbol='diamond'),
    name='SPIKE Anomaly',
    text=spikes['deviation_pct'].apply(lambda x: f"{x:.1f}%"),
    hovertemplate='<b>SPIKE</b><br>Date: %{x}<br>Sales: %{y:.0f}<br>Deviation: %{text}<extra></extra>'
))

fig.update_layout(
    title=' Agentic BI — Sales Anomaly Detection',
    xaxis_title='Week',
    yaxis_title='Weekly Sales ($)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500
)

fig.show()

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
print(repr(api_key))

None


In [22]:
import os
print(os.getcwd())

c:\startup approach\agenthood ai


In [18]:

import sys
!{sys.executable} -m pip install groq


   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   ---------------------------------------- 2/2 [groq]



In [32]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
print(" Groq client ready")

# Quick test
test = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say exactly this: Groq connected successfully."}]
)
print(test.choices[0].message.content)



 Groq client ready
Groq connected successfully.


In [ ]:
import time

def explain_anomaly(date, actual_sales, expected_sales, deviation_pct, anomaly_type):
    
    direction = "dropped" if anomaly_type == "DROP" else "spiked"
    abs_deviation = abs(deviation_pct)
    
    prompt = f"""You are a senior retail analytics expert at a CPG company.

A sales anomaly was detected:
- Date: {date.strftime('%B %d, %Y')}
- Actual Sales: ${actual_sales:,.0f}
- Expected Sales: ${expected_sales:,.0f}
- Sales {direction} by {abs_deviation:.1f}% from forecast
- Anomaly Type: {anomaly_type}

Respond in EXACTLY this format, two lines only, nothing else:
REASON: [one sentence explaining the most likely business reason]
ACTION: [one sentence recommending the single most important next step]"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.choices[0].message.content.strip()
    lines = text.split('\n')
    
    reason = "Unable to generate explanation"
    action = "Unable to generate recommendation"
    
    for line in lines:
        if line.startswith("REASON:"):
            reason = line.replace("REASON:", "").strip()
        elif line.startswith("ACTION:"):
            action = line.replace("ACTION:", "").strip()
    
    return reason, action

# Test on worst anomaly first
test_row = anomalies.sort_values('deviation_pct').iloc[0]

print(f"Date: {test_row['ds'].strftime('%B %d, %Y')}")
print(f"Actual: ${test_row['y']:,.0f} | Expected: ${test_row['yhat']:,.0f}")
print(f"Deviation: {test_row['deviation_pct']}%\n")

reason, action = explain_anomaly(
    test_row['ds'],
    test_row['y'],
    test_row['yhat'],
    test_row['deviation_pct'],
    test_row['anomaly_type']
)

print(f"REASON: {reason}")
print(f"ACTION: {action}")


Date: March 15, 2015
Actual: $2,729 | Expected: $11,277
Deviation: -75.8%

REASON: The 75.8% drop in sales on March 15, 2015, likely resulted from a point-of-sale system issue or a data feed error at the retailer's stores.
ACTION: Conduct a thorough investigation with the retailer to confirm the root cause and obtain corrected sales data to ensure accurate future forecasts.


In [26]:
print(f"Processing {len(anomalies)} anomalies through Llama...\n")

reasons = []
actions = []

for idx, row in anomalies.iterrows():
    reason, action = explain_anomaly(
        row['ds'],
        row['y'],
        row['yhat'],
        row['deviation_pct'],
        row['anomaly_type']
    )
    reasons.append(reason)
    actions.append(action)
    time.sleep(0.3)
    print(f"✓ {row['ds'].strftime('%Y-%m-%d')} | {row['anomaly_type']} {row['deviation_pct']:.1f}%")

anomalies = anomalies.copy()
anomalies['reason'] = reasons
anomalies['action'] = actions

print(f"\n Done! All {len(anomalies)} anomalies explained.")

Processing 33 anomalies through Llama...

✓ 2014-03-23 | SPIKE 228.0%
✓ 2014-07-27 | SPIKE 226.4%
✓ 2014-08-10 | SPIKE 127.6%
✓ 2014-09-14 | SPIKE 85.7%
✓ 2014-11-23 | SPIKE 42.9%
✓ 2015-03-15 | DROP -75.8%
✓ 2015-04-19 | SPIKE 101.9%
✓ 2015-09-13 | DROP -42.3%
✓ 2015-10-04 | DROP -68.3%
✓ 2015-12-13 | DROP -42.0%
✓ 2016-02-07 | SPIKE 93.8%
✓ 2016-03-06 | SPIKE 92.5%
✓ 2016-03-13 | SPIKE 79.7%
✓ 2016-03-20 | DROP -49.8%
✓ 2016-03-27 | DROP -74.1%
✓ 2016-04-17 | SPIKE 87.0%
✓ 2016-05-29 | SPIKE 120.5%
✓ 2016-09-25 | DROP -35.2%
✓ 2016-10-02 | SPIKE 62.2%
✓ 2016-10-30 | DROP -57.4%
✓ 2016-11-20 | DROP -42.7%
✓ 2017-01-22 | SPIKE 115.9%
✓ 2017-03-12 | DROP -54.5%
✓ 2017-03-26 | SPIKE 64.5%
✓ 2017-08-20 | SPIKE 80.3%
✓ 2017-08-27 | SPIKE 58.3%
✓ 2017-10-01 | DROP -51.6%
✓ 2017-10-22 | SPIKE 116.0%
✓ 2017-10-29 | DROP -59.5%
✓ 2017-11-05 | SPIKE 57.8%
✓ 2017-11-19 | SPIKE 47.3%
✓ 2017-12-03 | SPIKE 38.7%
✓ 2017-12-17 | DROP -52.4%

✅ Done! All 33 anomalies explained.


In [33]:
# Calculate summary stats for the header
total_weeks = len(results)
total_drops = len(anomalies[anomalies['anomaly_type'] == 'DROP'])
total_spikes = len(anomalies[anomalies['anomaly_type'] == 'SPIKE'])
worst_drop = anomalies[anomalies['anomaly_type'] == 'DROP']['deviation_pct'].min()
worst_drop_date = anomalies[anomalies['anomaly_type'] == 'DROP'].sort_values('deviation_pct').iloc[0]['ds'].strftime('%b %Y')
best_spike = anomalies[anomalies['anomaly_type'] == 'SPIKE']['deviation_pct'].max()
best_spike_date = anomalies[anomalies['anomaly_type'] == 'SPIKE'].sort_values('deviation_pct', ascending=False).iloc[0]['ds'].strftime('%b %Y')

date_start = results['ds'].min().strftime('%b %Y')
date_end = results['ds'].max().strftime('%b %Y')

print(f"""
📊 DASHBOARD SUMMARY
─────────────────────────────────────────
📅 Period Analysed  : {date_start} – {date_end}
📈 Weeks Analysed   : {total_weeks}
🔴 Drop Anomalies   : {total_drops}
🟠 Spike Anomalies  : {total_spikes}
⚠️  Worst Drop      : {worst_drop:.1f}% in {worst_drop_date}
🚀 Biggest Spike    : +{best_spike:.1f}% in {best_spike_date}
─────────────────────────────────────────
""")


📊 DASHBOARD SUMMARY
─────────────────────────────────────────
📅 Period Analysed  : Jan 2014 – Dec 2017
📈 Weeks Analysed   : 209
🔴 Drop Anomalies   : 13
🟠 Spike Anomalies  : 20
⚠️  Worst Drop      : -75.8% in Mar 2015
🚀 Biggest Spike    : +228.0% in Mar 2014
─────────────────────────────────────────



In [34]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import json

# ── Anomaly lookup for JavaScript ─────────────────────────────────────
anomaly_lookup = {}
for _, row in anomalies.iterrows():
    date_str = row['ds'].strftime('%Y-%m-%d')
    anomaly_lookup[date_str] = {
        'type': row['anomaly_type'],
        'actual': round(row['y'], 0),
        'expected': round(row['yhat'], 0),
        'deviation': round(row['deviation_pct'], 1),
        'reason': row['reason'],
        'action': row['action']
    }
anomaly_json = json.dumps(anomaly_lookup)

# ── Build figure ───────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.65, 0.35],
    subplot_titles=(
        "Sales Anomaly Detection — Click any dot for AI Insight", ""
    ),
    vertical_spacing=0.12
)

# Confidence band
fig.add_trace(go.Scatter(
    x=results['ds'].tolist() + results['ds'].tolist()[::-1],
    y=results['yhat_upper'].tolist() + results['yhat_lower'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(46, 117, 182, 0.12)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Expected Range',
    hoverinfo='skip'
), row=1, col=1)

# Forecast line
fig.add_trace(go.Scatter(
    x=results['ds'], y=results['yhat'],
    mode='lines',
    line=dict(color='rgba(46,117,182,0.5)', dash='dash', width=1.5),
    name='Forecast'
), row=1, col=1)

# Actual sales
fig.add_trace(go.Scatter(
    x=results['ds'], y=results['y'],
    mode='lines',
    line=dict(color='#1F4E79', width=2),
    name='Actual Sales'
), row=1, col=1)

# DROP anomalies
drops = anomalies[anomalies['anomaly_type'] == 'DROP']
fig.add_trace(go.Scatter(
    x=drops['ds'], y=drops['y'],
    mode='markers',
    marker=dict(color='red', size=11, symbol='circle',
                line=dict(color='darkred', width=1)),
    name='DROP',
    customdata=drops[['reason', 'action', 'deviation_pct', 'anomaly_type']].values,
    hovertemplate=(
        '<b>🔴 DROP Anomaly</b><br>'
        'Date: %{x}<br>Sales: $%{y:,.0f}<br>'
        'Deviation: %{customdata[2]:.1f}%<br>'
        '<i>Click for AI insight</i><extra></extra>'
    )
), row=1, col=1)

# SPIKE anomalies
spikes = anomalies[anomalies['anomaly_type'] == 'SPIKE']
fig.add_trace(go.Scatter(
    x=spikes['ds'], y=spikes['y'],
    mode='markers',
    marker=dict(color='orange', size=11, symbol='diamond',
                line=dict(color='darkorange', width=1)),
    name='SPIKE',
    customdata=spikes[['reason', 'action', 'deviation_pct', 'anomaly_type']].values,
    hovertemplate=(
        '<b>🟠 SPIKE Anomaly</b><br>'
        'Date: %{x}<br>Sales: $%{y:,.0f}<br>'
        'Deviation: %{customdata[2]:.1f}%<br>'
        '<i>Click for AI insight</i><extra></extra>'
    )
), row=1, col=1)

# Placeholder text
fig.add_trace(go.Scatter(
    x=[0.5], y=[0.5],
    mode='text',
    text=['👆 Click any anomaly dot above to see AI-generated insight'],
    textfont=dict(size=13, color='#888888'),
    showlegend=False,
    hoverinfo='skip'
), row=2, col=1)

fig.update_layout(
    height=750,
    template='plotly_white',
    title=dict(
        text='🤖 Agentic BI Demo — Sales Intelligence Platform',
        font=dict(size=18, color='#1F4E79')
    ),
    hovermode='closest',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1),
    clickmode='event+select'
)
fig.update_xaxes(title_text="Week", row=1, col=1)
fig.update_yaxes(title_text="Weekly Sales ($)", row=1, col=1)
fig.update_xaxes(visible=False, row=2, col=1)
fig.update_yaxes(visible=False, row=2, col=1, range=[0, 1])

# ── Build anomaly table rows ───────────────────────────────────────────
table_rows = ""
for _, row in anomalies.sort_values('deviation_pct').iterrows():
    color  = "#c0392b" if row['anomaly_type'] == 'DROP' else "#e67e22"
    badge  = f'<span style="background:{color};color:white;padding:2px 8px;border-radius:4px;font-size:11px;">{row["anomaly_type"]}</span>'
    arrow  = "▼" if row['anomaly_type'] == 'DROP' else "▲"
    abspct = abs(row['deviation_pct'])
    table_rows += f"""
        <tr style="border-bottom:1px solid #eee;">
          <td style="padding:8px 10px;white-space:nowrap;">{row['ds'].strftime('%b %d, %Y')}</td>
          <td style="padding:8px 10px;">{badge}</td>
          <td style="padding:8px 10px;color:{color};font-weight:bold;">{arrow} {abspct:.1f}%</td>
          <td style="padding:8px 10px;color:#333;">${row['y']:,.0f}</td>
          <td style="padding:8px 10px;color:#555;">${row['yhat']:,.0f}</td>
          <td style="padding:8px 10px;color:#1F4E79;font-size:12px;">{row['reason']}</td>
          <td style="padding:8px 10px;color:#1a7a3c;font-size:12px;">{row['action']}</td>
        </tr>"""

# ── Summary stats ──────────────────────────────────────────────────────
summary_html = f"""
<div style="max-width:960px;margin:20px auto;padding:0 20px;font-family:Arial,sans-serif;">

  <!-- Header stats bar -->
  <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:20px;">
    {"".join([
      f'<div style="flex:1;min-width:140px;background:{bg};border-radius:8px;padding:14px 18px;text-align:center;">'
      f'<div style="font-size:22px;font-weight:bold;color:{tc};">{val}</div>'
      f'<div style="font-size:11px;color:#666;margin-top:4px;">{label}</div></div>'
      for bg, tc, val, label in [
          ("#EAF2FB","#1F4E79", f"{total_weeks}", "Weeks Analysed"),
          ("#FDEDEC","#c0392b", f"{total_drops}", "Drop Anomalies"),
          ("#FEF9E7","#e67e22", f"{total_spikes}", "Spike Anomalies"),
          ("#FDEDEC","#c0392b", f"{worst_drop:.1f}%", f"Worst Drop ({worst_drop_date})"),
          ("#EAFAF1","#1a7a3c", f"+{best_spike:.1f}%", f"Biggest Spike ({best_spike_date})"),
      ]
    ])}
  </div>

  <!-- Insight panel (click target) -->
  <div id="insight-panel"
       style="margin-bottom:20px;padding:14px 18px;background:#f8f9fa;
              border-radius:6px;color:#888;text-align:center;font-size:13px;">
    👆 Click any red or orange dot on the chart to see the AI-generated insight
  </div>

  <!-- Anomaly table -->
  <h3 style="color:#1F4E79;margin-bottom:10px;">📋 All Anomalies — AI Insights Summary</h3>
  <div style="overflow-x:auto;">
    <table style="width:100%;border-collapse:collapse;font-size:13px;font-family:Arial,sans-serif;">
      <thead>
        <tr style="background:#1F4E79;color:white;">
          <th style="padding:10px;text-align:left;">Date</th>
          <th style="padding:10px;text-align:left;">Type</th>
          <th style="padding:10px;text-align:left;">Deviation</th>
          <th style="padding:10px;text-align:left;">Actual</th>
          <th style="padding:10px;text-align:left;">Expected</th>
          <th style="padding:10px;text-align:left;">🧠 AI Reason</th>
          <th style="padding:10px;text-align:left;">✅ Recommended Action</th>
        </tr>
      </thead>
      <tbody>{table_rows}</tbody>
    </table>
  </div>
</div>
"""

# ── JavaScript click handler ───────────────────────────────────────────
js_code = f"""
<script>
var anomalyData = {anomaly_json};
document.addEventListener('DOMContentLoaded', function() {{
    var plotDiv = document.querySelector('.plotly-graph-div');
    plotDiv.on('plotly_click', function(data) {{
        var point = data.points[0];
        var rawDate = point.x;
        var d = new Date(rawDate);
        var dateKey = d.toISOString().split('T')[0];
        var info = anomalyData[dateKey];
        if (!info) return;
        var color  = info.type === 'DROP' ? '#c0392b' : '#e67e22';
        var emoji  = info.type === 'DROP' ? '🔴' : '🟠';
        var arrow  = info.deviation < 0  ? '▼' : '▲';
        var absdev = Math.abs(info.deviation);
        document.getElementById('insight-panel').innerHTML = `
          <div style="background:#f8f9fa;border-left:5px solid ${{color}};
                      padding:16px 20px;border-radius:6px;text-align:left;">
            <div style="font-size:15px;font-weight:bold;color:${{color}};margin-bottom:10px;">
              ${{emoji}} ${{info.type}} — ${{dateKey}}
              &nbsp;|&nbsp; ${{arrow}} ${{absdev}}% from forecast
            </div>
            <div style="margin-bottom:10px;">
              <span style="background:#e8e8e8;padding:2px 8px;border-radius:4px;font-size:12px;">
                Actual</span>&nbsp;<strong>$${{info.actual.toLocaleString()}}</strong>
              &nbsp;&nbsp;
              <span style="background:#e8e8e8;padding:2px 8px;border-radius:4px;font-size:12px;">
                Expected</span>&nbsp;<strong>$${{info.expected.toLocaleString()}}</strong>
            </div>
            <div style="margin-bottom:8px;">
              <span style="color:#1F4E79;font-weight:bold;">🧠 AI Reason:</span>
              <span style="color:#333;"> ${{info.reason}}</span>
            </div>
            <div>
              <span style="color:#1a7a3c;font-weight:bold;">✅ Recommended Action:</span>
              <span style="color:#333;"> ${{info.action}}</span>
            </div>
          </div>`;
    }});
}});
</script>
"""

# ── Assemble final HTML ────────────────────────────────────────────────
html_str   = fig.to_html(include_plotlyjs='cdn', full_html=True)
html_final = html_str.replace(
    '</body>',
    summary_html + js_code + '</body>'
)

with open('agentic_bi_demo.html', 'w', encoding='utf-8') as f:
    f.write(html_final)

print(" Full dashboard saved — agentic_bi_demo.html")

import webbrowser
webbrowser.open('agentic_bi_demo.html')

✅ Full dashboard saved — agentic_bi_demo.html


True

In [29]:
import webbrowser
webbrowser.open('agentic_bi_demo.html')

True

In [30]:
anomalies[['ds', 'y', 'yhat', 'deviation_pct', 'anomaly_type', 'reason', 'action']]\
    .rename(columns={
        'ds': 'Week',
        'y': 'Actual_Sales',
        'yhat': 'Expected_Sales',
        'deviation_pct': 'Deviation_%',
        'anomaly_type': 'Type',
        'reason': 'AI_Reason',
        'action': 'AI_Recommended_Action'
    })\
    .to_csv('anomaly_insights.csv', index=False)

print(" Saved anomaly_insights.csv")

 Saved anomaly_insights.csv
